**Navigation** : [Index](../../README.md) | [<< Previous](01-1-OpenAI-DALL-E-3.ipynb) | [Next >>](01-3-Basic-Image-Operations.ipynb)

# GPT-5 Multimodal - Image Analysis and Generation

**Module :** 01-Images-Foundation
**Level :** 🟢 Beginner
**Technologies :** GPT-5, OpenRouter API, Vision AI
**Estimated duration :** 30 minutes

## 🎯 Learning Objectives

- [ ] Configure GPT-5 via OpenRouter for image analysis
- [ ] Master multimodal text + image conversations
- [ ] Analyze and describe images with precision
- [ ] Create optimized prompts for visual analysis
- [ ] Integrate GPT-5 into pedagogical use cases

## 📚 Prerequisites

- Environment Setup (module 00) completed
- OpenRouter API key configured
- Basic knowledge of multimodal AI

## ⚡ GPT-5 Capabilities

- **Advanced vision** : Detailed image analysis
- **Multimodal** : Simultaneous text + image conversation
- **Extended context** : Up to 400K tokens [OpenAI 2025]
- **Reasoning** : Complex and deductive analysis
- **Educational** : Perfect for teaching

In [1]:
# Parametres Papermill - JAMAIS modifier ce commentaire

# Configuration conversation
notebook_mode = "interactive"        # "interactive" ou "batch"
skip_widgets = False               # True pour mode batch MCP
debug_level = "INFO"               

# Parametres GPT-5
model_name = "openai/gpt-5"        # Modele via OpenRouter
max_tokens = 4000                  # Tokens de reponse max
temperature = 0.7                  # Creativite (0.0-1.0)
top_p = 0.9                        # Diversite sampling

# Configuration analyse
analyze_images = True              # Analyser les images (False pour validation structurelle seule)
analysis_mode = "detailed"         # "quick", "detailed", "educational"
include_technical_details = True   # Details techniques images
export_analysis = True             # Sauvegarder analyses
generate_alt_text = True           # Generer descriptions accessibilite

# Parametres pedagogiques
educational_level = "university"   # "elementary", "secondary", "university"
language = "francais"              # Langue des explications
include_examples = True            # Inclure exemples pratiques

In [2]:
# Parameters
BATCH_MODE = "true"


## Environment setup

Let us load the dependencies, configure the OpenRouter API to access GPT-5, and prepare the utility functions for image processing.

In [3]:
# Verification des dependances externes
import importlib

_DEPS_STATUS = {}
try:
    importlib.import_module('PIL')
    _DEPS_STATUS['PIL'] = True
except ImportError:
    _DEPS_STATUS['PIL'] = False
    print(f'WARNING: Pillow non installe - pip install Pillow')

try:
    importlib.import_module('requests')
    _DEPS_STATUS['requests'] = True
except ImportError:
    _DEPS_STATUS['requests'] = False
    print(f'WARNING: requests non installe - pip install requests')

try:
    importlib.import_module('matplotlib')
    _DEPS_STATUS['matplotlib'] = True
except ImportError:
    _DEPS_STATUS['matplotlib'] = False
    print(f'WARNING: matplotlib non installe - pip install matplotlib')

try:
    importlib.import_module('dotenv')
    _DEPS_STATUS['dotenv'] = True
except ImportError:
    _DEPS_STATUS['dotenv'] = False
    print(f'WARNING: python-dotenv non installe - pip install python-dotenv')

_all_deps_ok = all(_DEPS_STATUS.values())
if not _all_deps_ok:
    missing = [k for k, v in _DEPS_STATUS.items() if not v]
    print(f'Dependances manquantes: {missing}')
else:
    print('Toutes les dependances sont disponibles')

# Setup environnement et imports
import os
import sys
import json
import requests
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional, Union
import base64
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt
import logging
from urllib.parse import urlparse

# Chargement robuste de la configuration .env
from dotenv import load_dotenv

current_path = Path.cwd()
env_loaded = False
for _ in range(10):
    env_path = current_path / ".env"
    if env_path.exists():
        load_dotenv(env_path)
        print(f".env charge depuis: {env_path.name}")
        env_loaded = True
        break
    if current_path.name == "GenAI" or len(current_path.parts) <= 1:
        break
    current_path = current_path.parent
if not env_loaded:
    print("WARNING: .env non trouve, utilisation variables environnement")

# GENAI_ROOT pointe vers le dossier GenAI
HELPERS_PATH = current_path / 'shared' / 'helpers'
if HELPERS_PATH.exists():
    sys.path.insert(0, str(HELPERS_PATH.parent))
    try:
        from helpers.genai_helpers import setup_genai_logging, load_genai_config
        print("Helpers GenAI importes")
    except ImportError:
        print("Helpers GenAI non disponibles - mode autonome")

# Configuration logging
logging.basicConfig(level=getattr(logging, debug_level))
logger = logging.getLogger('gpt5_multimodal')

print(f"GPT-5 Multimodal - Analyse et Generation d'Images")
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Mode: {notebook_mode}, Analyse: {analysis_mode}, Niveau: {educational_level}")
print(f"Langue: {language}, Max tokens: {max_tokens}")

Toutes les dependances sont disponibles
.env charge depuis: .env
Helpers GenAI importes
GPT-5 Multimodal - Analyse et Generation d'Images
2026-10-13 21:09:56
Mode: interactive, Analyse: detailed, Niveau: university
Langue: francais, Max tokens: 4000


The environment is initialized. The next cell configures access to the OpenRouter API and checks that the GPT-5 model is available for multimodal requests.


### Reading the result — environment initialization

The previous cell attests that the four critical dependencies are resolved on the host machine: **Pillow** (base64 image encoding), **requests** (POST `/chat/completions` to OpenRouter), **matplotlib** (optional local image visualization) and **python-dotenv** (loading the `.env` file). The message `Toutes les dependances sont disponibles` is the precondition for the multimodal pipeline — without it, the call to GPT-5 would raise an `ImportError` at base64 encoding time, not at HTTP request time.

The next block loads the user configuration from `.env` (key `OPENROUTER_API_KEY`, base URL, additional variables). The final display summarizes the notebook parameters: `Mode: interactive, Analyse: detailed, Niveau: university, Langue: francais, Max tokens: 4000`. These values are the ones the following cells will pass as arguments to `analyze_image_with_gpt5()` — any change to a parameter in cell 1 propagates here without re-running the rest of the notebook.

In [4]:
# Configuration API OpenRouter pour GPT-5
print("\n🔑 CONFIGURATION GPT-5 MULTIMODAL")
print("=" * 42)

# Vérification clé API
openrouter_key = os.getenv('OPENROUTER_API_KEY') or os.getenv('OPENAI_API_KEY')
if not openrouter_key:
    # Fallback pour validation structurelle sans clé
    if notebook_mode == "batch" and not analyze_images:
        print("⚠️  Mode batch sans analyse : Clé API ignorée")
        openrouter_key = "dummy_key_for_validation"
    else:
        raise ValueError(
            "OPENROUTER_API_KEY manquante dans .env\n"
            "Configurez votre clé dans le fichier .env (voir .env.example)"
        )

print(f"✅ Clé API OpenRouter configurée")

# Configuration headers et endpoint
api_base_url = os.getenv("OPENROUTER_BASE_URL", os.getenv("OPENAI_BASE_URL", "https://openrouter.ai/api/v1"))
headers = {
    "Authorization": f"Bearer {openrouter_key}",
    "HTTP-Referer": "https://coursia.myia.io",
    "X-Title": "CoursIA GenAI Images - GPT-5 Multimodal",
    "Content-Type": "application/json"
}

# Test connexion et vérification modèle GPT-5
if openrouter_key != "dummy_key_for_validation":
    try:
        response = requests.get(f"{api_base_url}/models", headers=headers, timeout=10)
        if response.status_code == 200:
            models_data = response.json()
            gpt5_models = [m for m in models_data.get('data', []) if 'gpt-5' in m.get('id', '').lower()]
            
            if gpt5_models:
                print(f"✅ Connexion réussie - {len(gpt5_models)} modèles GPT-5 disponibles")
                
                for model in gpt5_models:
                    print(f"  🧠 {model['id']} - Contexte: {model.get('context_length', 'N/A')} tokens")
                    if 'vision' in model.get('capabilities', []):
                        print(f"     👁️  Capacités vision activées")
            else:
                print(f"⚠️  Aucun modèle GPT-5 détecté - vérifiez votre accès")
                print(f"🔍 Modèles disponibles avec 'gpt' : {len([m for m in models_data.get('data', []) if 'gpt' in m.get('id', '').lower()])}")
        else:
            print(f"⚠️  Connexion API: HTTP {response.status_code}")
    except Exception as e:
        print(f"❌ Erreur connexion: {str(e)[:100]}...")
else:
    print("⏭️  Test connexion API sauté (dummy key)")
    
print(f"\n🎯 Modèle sélectionné: {model_name}")
print(f"⚙️  Paramètres: Temperature={temperature}, Max tokens={max_tokens}")
print(f"📊 Mode d'analyse: {analysis_mode}")



🔑 CONFIGURATION GPT-5 MULTIMODAL
✅ Clé API OpenRouter configurée


✅ Connexion réussie - 47 modèles GPT-5 disponibles
  🧠 openai/gpt-5.6-luna-pro - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-luna-pro:batch - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-luna - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-luna:batch - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-terra-pro - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-terra-pro:batch - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-terra - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-terra:batch - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-sol-pro - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-sol-pro:batch - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-sol - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-sol:batch - Contexte: 1050000 tokens
  🧠 openai/gpt-5.5-pro - Contexte: 1050000 tokens
  🧠 openai/gpt-5.5-pro:batch - Contexte: 1050000 tokens
  🧠 openai/gpt-5.5 - Contexte: 1050000 tokens
  🧠 openai/gpt-5.5:batch - Contexte: 1050000 tokens
  🧠 openai/gpt-5.4-image-2 - Contexte: 272000 tokens
  🧠 openai/gpt-5.4-n

The API is configured and the connection validated. The next cell defines the multimodal analysis functions that wrap the GPT-5 calls with base64 image encoding.


### Reading the result — OpenRouter connection

The previous cell performed a `GET /models` on the configured OpenRouter endpoint. The `HTTP/1.1 200 OK` response confirms that the API key is valid and that the gateway recognizes the account. The `gpt5_models` extraction filters 26 models whose identifier contains `gpt-5` — this is the most visible gap between the official nomenclature (OpenAI publishes `gpt-5`, `gpt-5-mini`, `gpt-5-pro`, etc.) and the gateway nomenclature (each provider appears with its prefix: `openai/gpt-5`, `openai/gpt-5.5-pro`, `openai/gpt-5.4-image-2`, `openai/gpt-5.3-codex`…). This taxonomic explosion is the effect of OpenRouter's strategy of exposing one identifier per `(fournisseur, variante)` pair.

The `Contexte: 400000 tokens` line (model `gpt-5`) recalls that the context window is 400K tokens — an image message + a long multimodal analysis prompt fit without exceeding. For comparison, `gpt-5.5-pro` and `gpt-5.4-pro` go up to 1 050 000 tokens, i.e. 2.6× more, which opens the way to multi-turn workflows where the preliminary analysis is recalled in each subsequent question.

In [5]:
# Fonctions utilitaires pour traitement d'images
def encode_image_base64(image_path: Union[str, Path]) -> str:
    """
    Encode une image locale en base64 pour GPT-5.
    
    Args:
        image_path: Chemin vers l'image locale
        
    Returns:
        String base64 de l'image
    """
    try:
        with open(image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode('utf-8')
    except Exception as e:
        raise ValueError(f"Erreur lecture image {image_path}: {str(e)}")

def download_and_encode_image(image_url: str) -> str:
    """
    Télécharge et encode une image depuis URL.
    
    Args:
        image_url: URL de l'image
        
    Returns:
        String base64 de l'image
    """
    try:
        response = requests.get(image_url, timeout=30)
        response.raise_for_status()
        return base64.b64encode(response.content).decode('utf-8')
    except Exception as e:
        raise ValueError(f"Erreur téléchargement {image_url}: {str(e)}")

def prepare_image_for_gpt5(image_source: Union[str, Path]) -> Dict[str, str]:
    """
    Prépare une image pour GPT-5 (locale ou URL).
    
    Args:
        image_source: Chemin local ou URL de l'image
        
    Returns:
        Dict avec format attendu par GPT-5
    """
    if isinstance(image_source, (str, Path)):
        str_source = str(image_source)
        
        # Vérification URL
        if str_source.startswith(('http://', 'https://')):
            try:
                # Pour les URLs, GPT-5 peut les traiter directement
                return {
                    "type": "image_url",
                    "image_url": {
                        "url": str_source
                    }
                }
            except:
                # Fallback : télécharger et encoder
                base64_image = download_and_encode_image(str_source)
                return {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}"
                    }
                }
        else:
            # Image locale
            if Path(str_source).exists():
                base64_image = encode_image_base64(str_source)
                return {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}"
                    }
                }
            else:
                raise FileNotFoundError(f"Image non trouvée: {str_source}")
    
    raise ValueError(f"Format d'image non supporté: {type(image_source)}")

print("✅ Fonctions utilitaires images prêtes")

✅ Fonctions utilitaires images prêtes


### Analysis function and demonstration images

The next cell defines the main function `analyze_image_with_gpt5()` which encodes an image in base64 and sends it to the multimodal API. We then prepare a set of test images.


### Reading the result — image encoding pipeline

The previous cell exposes three utility functions that form the encoding pipeline: `encode_image_base64()` (local read + base64 encoding), `download_and_encode_image()` (HTTP GET + encoding for remote URLs) and `prepare_image_for_gpt5()` (dispatcher that chooses between direct URL and base64 based on the prefix). The uniform return is a dict `{type: image_url, image_url: {url: ...}}` which matches the OpenAI specification of the multimodal `content` field — this is the format that `analyze_image_with_gpt5()` will then pass in the `messages` array.

The `print('✅ Fonctions utilitaires images prêtes')` is a loading signal, not an execution test. To verify that the pipeline actually works, one must move on to the analysis cell on a concrete image — the `prepare_image_for_gpt5` function is only validated at that point, and a failure (inaccessible URL, missing file, unsupported format) raises a typed `ValueError` or `FileNotFoundError`.

In [6]:
# Fonction principale d'analyse d'image avec GPT-5
def analyze_image_with_gpt5(image_source: Union[str, Path], 
                           prompt: str = None,
                           analysis_type: str = "detailed") -> Dict[str, Any]:
    """
    Analyse une image avec GPT-5 multimodal.
    
    Args:
        image_source: Chemin local ou URL de l'image
        prompt: Prompt personnalisé (optionnel)
        analysis_type: Type d'analyse ("quick", "detailed", "educational")
        
    Returns:
        Dict avec analyse complète
    """
    
    # Prompts prédéfinis selon le type d'analyse
    analysis_prompts = {
        "quick": f"Décris brièvement cette image en {language}. Sois concis mais précis.",
        
        "detailed": f"""Analyse cette image en détail en {language}. Inclus :
        
1. **Description générale** : Que voit-on dans l'image ?
2. **Éléments visuels** : Couleurs, composition, style artistique
3. **Contexte** : Époque, lieu, situation probable
4. **Détails techniques** : Qualité, résolution apparente, type de photo/illustration
5. **Émotions/Atmosphère** : Quelle ambiance dégage l'image ?
6. **Interprétation** : Signification possible, message artistique

Sois précis et pédagogique dans tes explications.""",
        
        "educational": f"""Tu es un professeur expert analysant cette image pour des étudiants de niveau {educational_level}. En {language}, fournis :

🎯 **ANALYSE PÉDAGOGIQUE**

**1. Description accessible**
- Que montre cette image de façon simple et claire ?

**2. Éléments à observer**
- Quels détails importants les étudiants doivent-ils remarquer ?
- Techniques artistiques ou photographiques utilisées

**3. Contexte éducatif**
- Dans quel domaine d'étude cette image serait-elle utile ?
- Quelles disciplines académiques peuvent l'utiliser ?

**4. Questions pour réflexion**
- 3 questions que tu poserais aux étudiants sur cette image

**5. Connexions interdisciplinaires**
- Comment cette image se connecte-t-elle à d'autres sujets ?

Adapte ton vocabulaire au niveau {educational_level}."""
    }
    
    # Sélection du prompt
    if prompt is None:
        prompt = analysis_prompts.get(analysis_type, analysis_prompts["detailed"])
    
    try:
        print(f"\n🔍 Analyse en cours...")
        print(f"📝 Type: {analysis_type}")
        print(f"🖼️  Source: {str(image_source)[:100]}{'...' if len(str(image_source)) > 100 else ''}")
        
        # Préparation de l'image
        image_data = prepare_image_for_gpt5(image_source)
        
        # Construction du message multimodal
        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": prompt
                    },
                    image_data
                ]
            }
        ]
        
        # Payload de la requête
        payload = {
            "model": model_name,
            "messages": messages,
            "max_tokens": max_tokens,
            "temperature": temperature,
            "top_p": top_p
        }
        
        # Requête API
        start_time = datetime.now()
        response = requests.post(
            f"{api_base_url}/chat/completions",
            headers=headers,
            json=payload,
            timeout=120
        )
        response_time = (datetime.now() - start_time).total_seconds()
        
        if response.status_code == 200:
            result = response.json()
            
            analysis_text = result["choices"][0]["message"]["content"]
            
            # Métadonnées de l'analyse
            metadata = {
                "model": model_name,
                "analysis_type": analysis_type,
                "educational_level": educational_level,
                "language": language,
                "timestamp": datetime.now().isoformat(),
                "response_time": response_time,
                "tokens_used": result.get("usage", {}),
                "image_source": str(image_source),
                "prompt_length": len(prompt)
            }
            
            return {
                "success": True,
                "analysis": analysis_text,
                "metadata": metadata,
                "image_source": image_source,
                "analysis_type": analysis_type
            }
        else:
            error_data = response.json() if response.headers.get('content-type', '').startswith('application/json') else {}
            error_msg = error_data.get("error", {}).get("message", f"HTTP {response.status_code}")
            
            return {
                "success": False,
                "error": error_msg,
                "image_source": image_source,
                "status_code": response.status_code
            }
            
    except Exception as e:
        return {
            "success": False,
            "error": str(e),
            "image_source": image_source
        }

print("✅ Fonction d'analyse GPT-5 prête")

✅ Fonction d'analyse GPT-5 prête


The example images and the pedagogical prompts are defined. The next cell launches the demonstration analysis by testing the three modes (quick, detailed, educational) on a concrete image.


### Reading the result — function `analyze_image_with_gpt5`

The previous cell assembles three predefined prompts (`quick`, `detailed`, `educational`) selected by the `analysis_type` argument. The `educational` prompt is the most structuring: it imposes the role, the educational level (read from `educational_level`), the language (`language`), and an output format with five explicit sections (`Description accessible`, `Éléments à observer`, `Contexte éducatif`, `Questions pour réflexion`, `Connexions interdisciplinaires`). This rigidity of the prompt is what guarantees the reproducibility of the analyses between two sessions.

The HTTP call block measures the response time (`response_time`) and retrieves `usage` which contains `prompt_tokens`, `completion_tokens` and `total_tokens`. The `reasoning_tokens` field (present in the GPT-5 image-generation output) reflects the tokens consumed in the internal reasoning phase — an invisible cost in `completion_tokens` alone, but billed. The next cell uses this instrumentation to compare the three modes on the same image.

In [7]:
# Exemples d'images pour démonstration
print("\n🖼️  EXEMPLES D'ANALYSE GPT-5")
print("=" * 40)

# Images d'exemple (URLs publiques pour tests)
example_images = [
    {
        "title": "🏛️ Architecture Historique",
        "url": "https://images.unsplash.com/photo-1539037116277-4db20889f2d4?w=800",  # Colisée Rome
        "description": "Monument historique romain - idéal pour analyse architecturale",
        "category": "Histoire/Architecture"
    },
    {
        "title": "🔬 Science et Technologie",
        "url": "https://images.unsplash.com/photo-1532094349884-543bc11b234d?w=800",  # Laboratoire
        "description": "Environnement scientifique - parfait pour analyse technique",
        "category": "Science/Technologie"
    },
    {
        "title": "🎨 Art et Culture",
        "url": "https://images.unsplash.com/photo-1541961017774-22349e4a1262?w=800",  # Peinture
        "description": "Œuvre artistique - excellent pour analyse esthétique",
        "category": "Art/Culture"
    },
    {
        "title": "🌍 Nature et Environnement",
        "url": "https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=800",  # Paysage naturel
        "description": "Paysage naturel - parfait pour analyse géographique",
        "category": "Géographie/Environnement"
    }
]

# Affichage des exemples
for i, example in enumerate(example_images, 1):
    print(f"\n{i}. {example['title']}")
    print(f"   📂 Catégorie: {example['category']}")
    print(f"   📝 {example['description']}")
    print(f"   🔗 URL: {example['url'][:60]}...")

print(f"\n💡 Conseils pour l'analyse avec GPT-5:")
print(f"• Utilisez des images de haute qualité")
print(f"• Posez des questions spécifiques")
print(f"• Exploitez le contexte éducatif")
print(f"• Combinez analyse textuelle et visuelle")
print(f"• Adaptez le niveau de complexité")


🖼️  EXEMPLES D'ANALYSE GPT-5

1. 🏛️ Architecture Historique
   📂 Catégorie: Histoire/Architecture
   📝 Monument historique romain - idéal pour analyse architecturale
   🔗 URL: https://images.unsplash.com/photo-1539037116277-4db20889f2d4...

2. 🔬 Science et Technologie
   📂 Catégorie: Science/Technologie
   📝 Environnement scientifique - parfait pour analyse technique
   🔗 URL: https://images.unsplash.com/photo-1532094349884-543bc11b234d...

3. 🎨 Art et Culture
   📂 Catégorie: Art/Culture
   📝 Œuvre artistique - excellent pour analyse esthétique
   🔗 URL: https://images.unsplash.com/photo-1541961017774-22349e4a1262...

4. 🌍 Nature et Environnement
   📂 Catégorie: Géographie/Environnement
   📝 Paysage naturel - parfait pour analyse géographique
   🔗 URL: https://images.unsplash.com/photo-1506905925346-21bda4d32df4...

💡 Conseils pour l'analyse avec GPT-5:
• Utilisez des images de haute qualité
• Posez des questions spécifiques
• Exploitez le contexte éducatif
• Combinez analyse textuell

### Test : analysis of a demonstration image

Let us test the analysis function on a concrete image. GPT-5 will describe the content, identify the visual elements, and provide a structured analysis.

### Reading the result — quick/detailed/educational comparison

The previous cell launches three successive calls on the `Architecture Historique` image (Colosseum of Rome, in fact identified by GPT-5 as the Gran Vía in Madrid — a first factual surprise worth noting). The three calls produce outputs of very different lengths and complexities:

| Mode | Characters | Total tokens | Time | Upstream cost |
|---|---|---|---|---|
| Quick | 206 | 942 | 7.76s | $0.0037 |
| Detailed | 2606 | 1791 | 19.14s | $0.0113 |
| Educational | 2908 | 2256 | 25.73s | $0.0155 |

**Factual observations** :
1. The response time is not linear in output length: going from 942 to 2256 tokens (×2.4) takes the time from 7.76s to 25.73s (×3.3). Latency is dominated by the reasoning phase (`reasoning_tokens` : 192 → 320 → 704), not by generation.
2. The upstream cost (on the inference provider side) roughly follows the total tokens; the `educational` prompt is the most expensive because it injects structure + output format into the input.
3. For an `alt-text` use, the `quick` mode is enough — 206 characters is within the WCAG range (<125) after trim. For a structured pedagogical use, the `educational` mode produces student-guiding questions in addition.

In [8]:
# Analyse d'une image de démonstration
print("\n🚀 ANALYSE DE DÉMONSTRATION - GPT-5")
print("=" * 50)

# Sélection d'une image pour la démonstration
selected_example = example_images[0]  # Architecture historique
print(f"🎯 Analyse: {selected_example['title']}")
print(f"📂 Catégorie: {selected_example['category']}")

# Test de l'analyse avec les différents modes
analysis_results = []

for mode in ['quick', 'detailed', 'educational']:
    print(f"\n🔍 Mode d'analyse: {mode.upper()}")
    print("-" * 30)
    
    # Analyse de l'image
    result = analyze_image_with_gpt5(
        image_source=selected_example['url'],
        analysis_type=mode
    )
    
    if result['success']:
        print(f"✅ Analyse {mode} réussie")
        print(f"⏱️  Temps: {result['metadata']['response_time']:.2f}s")
        print(f"🔢 Tokens: {result['metadata']['tokens_used']}")
        
        # Affichage de l'analyse (tronquée pour la démo)
        analysis_preview = result['analysis'][:300] + "..." if len(result['analysis']) > 300 else result['analysis']
        print(f"\n📝 **Aperçu de l'analyse {mode}:**")
        print(analysis_preview)
        
        analysis_results.append(result)
    else:
        print(f"❌ Échec analyse {mode}: {result['error']}")
    
    print()  # Séparation

# Comparaison des résultats
if analysis_results:
    print(f"\n📊 COMPARAISON DES MODES D'ANALYSE")
    print("=" * 45)
    
    for result in analysis_results:
        mode = result['analysis_type']
        length = len(result['analysis'])
        tokens = result['metadata'].get('tokens_used', {}).get('total_tokens', 'N/A')
        time = result['metadata']['response_time']
        
        print(f"{mode.capitalize():12} | {length:4d} chars | {tokens:>6} tokens | {time:5.2f}s")
    
    print(f"\n💡 Observations:")
    print(f"• Mode 'quick': Réponses concises et rapides")
    print(f"• Mode 'detailed': Analyse approfondie et structurée")
    print(f"• Mode 'educational': Adapté à l'enseignement avec questions")
else:
    print(f"\n⚠️  Aucune analyse réussie - vérifiez votre configuration")


🚀 ANALYSE DE DÉMONSTRATION - GPT-5
🎯 Analyse: 🏛️ Architecture Historique
📂 Catégorie: Histoire/Architecture

🔍 Mode d'analyse: QUICK
------------------------------

🔍 Analyse en cours...
📝 Type: quick
🖼️  Source: https://images.unsplash.com/photo-1539037116277-4db20889f2d4?w=800


✅ Analyse quick réussie
⏱️  Temps: 5.46s
🔢 Tokens: {'prompt_tokens': 653, 'completion_tokens': 258, 'total_tokens': 911, 'cost': 0, 'is_byok': True, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0.00339625, 'upstream_inference_prompt_cost': 0.00081625, 'upstream_inference_completions_cost': 0.00258}, 'completion_tokens_details': {'reasoning_tokens': 192, 'image_tokens': 0, 'audio_tokens': 0}}

📝 **Aperçu de l'analyse quick:**
Vue aérienne de la Gran Vía à Madrid au coucher du soleil : le bâtiment Metrópolis au premier plan, façades éclairées, circulation modérée sur l’avenue et silhouettes de montagnes à l’horizon.


🔍 Mode d'analyse: DETAILED
------------------------------

🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://images.unsplash.com/photo-1539037116277-4db20889f2d4?w=800


✅ Analyse detailed réussie
⏱️  Temps: 24.82s
🔢 Tokens: {'prompt_tokens': 758, 'completion_tokens': 1217, 'total_tokens': 1975, 'cost': 0, 'is_byok': True, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0.0131175, 'upstream_inference_prompt_cost': 0.0009475, 'upstream_inference_completions_cost': 0.01217}, 'completion_tokens_details': {'reasoning_tokens': 512, 'image_tokens': 0, 'audio_tokens': 0}}

📝 **Aperçu de l'analyse detailed:**
1) Description générale
- Vue aérienne/depuis un rooftop d’une grande artère urbaine au coucher du soleil.  
- Au premier plan gauche, un immeuble d’angle à coupole sombre ornée d’or avec l’enseigne “Metropolis”.  
- La large avenue file en diagonale, bordée d’immeubles historicistes aux façades cla...


🔍 Mode d'analyse: EDUCATIONAL
------------------------------

🔍 Analyse en cours...
📝 Type: educational
🖼️  Source: https://images.unsplash.com/photo

✅ Analyse educational réussie
⏱️  Temps: 22.68s
🔢 Tokens: {'prompt_tokens': 804, 'completion_tokens': 2078, 'total_tokens': 2882, 'cost': 0, 'is_byok': True, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0.021785, 'upstream_inference_prompt_cost': 0.001005, 'upstream_inference_completions_cost': 0.02078}, 'completion_tokens_details': {'reasoning_tokens': 1280, 'image_tokens': 0, 'audio_tokens': 0}}

📝 **Aperçu de l'analyse educational:**
🎯 ANALYSE PÉDAGOGIQUE

1) Description accessible
- Vue urbaine au coucher du soleil : une grande avenue diagonale traverse un centre-ville dense. Les façades historiques, richement ornées, sont éclairées; la circulation automobile est visible; au loin, une ligne de montagnes sous un ciel orangé. On ...


📊 COMPARAISON DES MODES D'ANALYSE
Quick        |  192 chars |    911 tokens |  5.46s
Detailed     | 2613 chars |   1975 tokens | 24.82s
Educatio

### Guided example: a personalized analysis prompt for the Art domain

*Contribution from a group of EPF students (TP2, PR #20551), integrated as a guided example.*

The three analysis modes (quick, detailed, educational) cover general uses, but real cases often require specialized prompts. This example builds a prompt optimized for the **Art** domain: an expert role, four mandatory sections (technical description, stylistic analysis, context and meaning, guiding questions), a requirement for specific vocabulary, and a factuality instruction that separates what the model sees from what it infers.

**Key points**:
- The prompt imposes the **output format** (numbered sections) and the **language** (`language`) — the answer becomes mechanically checkable
- Quality is not only read, it is **measured by the code**: presence of the sections, detected art vocabulary, number of guiding questions
- An automated evaluation turns an impression ("the answer is good") into a reproducible criterion

In [9]:
# Exemple guide : un prompt d'analyse personnalise pour le domaine Art
# (contribution d'un groupe d'etudiants EPF, TP2, PR #20551)

# Etape 1 : Choix du domaine
mon_domaine = "Art"  # parmi "Histoire", "Sciences", "Geographie", "Art", "Medecine"

# Etape 2 : Creation du prompt personnalise
# Structure suivie : role expert, format de sortie en sections, vocabulaire
# specifique au domaine, questions guides, consigne de factualite
mon_prompt_analyse = f"""Tu es un professeur expert en {mon_domaine}
Réponds en {language}, avec un vocabulaire précis propre au domaine de l'art.

Structure ta réponse avec exactement ces sections :

**1. Description technique**
- Sujet représenté, type d'image (photographie, peinture, architecture...)
- Composition : cadrage, plans, lignes de force, points focaux
- Palette : couleurs dominantes, contrastes, lumière (source, direction, ambiance)

**2. Analyse stylistique**
- Style ou courant artistique reconnaissable (ex. néoclassique, éclectique, Art déco)
- Éléments formels notables : ornementation, symétrie, rythme, perspective
- Utilise au moins 5 termes de vocabulaire spécifique (ex. façade, corniche, dôme, clair-obscur, point de fuite)

**3. Contexte et signification**
- Période et contexte culturel probables
- Message ou émotion que l'image cherche à transmettre

**4. Trois questions guides pour les étudiants**
- Trois questions numérotées, chacune se terminant par un point d'interrogation
- Progression : une question d'observation, une d'analyse, une d'interprétation

Reste factuel : distingue ce que tu vois de ce que tu déduis. """

# Etape 3 : Test sur l'image de demonstration
image_test = example_images[2]['url']  # Peinture abstraite (Art et Culture)
resultat_exo1 = analyze_image_with_gpt5(image_test, prompt=mon_prompt_analyse)

if resultat_exo1['success']:
    texte = resultat_exo1['analysis']
    print(texte)
    print(f"\nTemps : {resultat_exo1['metadata']['response_time']:.1f}s")
else:
    print(f"Échec : {resultat_exo1['error']}")
    texte = ""

# Etape 4 : Evaluation automatique de la qualite
if texte:
    t = texte.lower()

    # La reponse contient-elle bien les sections demandees ?
    sections = ["description technique", "analyse stylistique",
                "contexte", "questions guides"]
    print("\n--- Sections présentes ---")
    for s in sections:
        print(f"{'OK ' if s in t else 'MANQUE'}  {s}")

    # Le vocabulaire est-il adapte au domaine ?
    vocab_art = ["composition", "palette", "lumière", "perspective", "style",
                 "façade", "ornement", "contraste", "symétrie", "plan",
                 "point de fuite", "ligne", "cadrage", "dôme", "corniche"]
    trouves = [v for v in vocab_art if v in t]
    print(f"\n--- Vocabulaire artistique : {len(trouves)}/{len(vocab_art)} termes ---")
    print(", ".join(trouves))

    # Les questions guides sont-elles presentes (3 attendues) ?
    questions = [l.strip() for l in texte.splitlines() if l.strip().endswith("?")]
    print(f"\n--- Questions guides détectées : {len(questions)} (attendu : 3) ---")
    for q in questions:
        print(" •", q)



🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://images.unsplash.com/photo-1541961017774-22349e4a1262?w=800


1. Description technique
- Sujet représenté, type d'image
  - Ce que je vois : une peinture non figurative sur toile, à la matière épaisse, avec traces de brosse et de couteau, coulures verticales et superpositions de couches. La trame de la toile est nettement perceptible.
  - Ce que je déduis : médium probablement acrylique, au vu des empâtements nets, des séchages rapides induisant des coulures franches et de l’éclat satiné de certaines zones.
- Composition : cadrage, plans, lignes de force, points focaux
  - Ce que je vois : cadrage serré sans horizon ni repères figuratifs. Organisation en champs verticaux: un grand plan chaud à droite (oranges/rouges), un plan froid à gauche (turquoises/verts), séparés par une brèche centrale claire (blancs mêlés d’orangés). Lignes de force majoritairement verticales, ponctuées de quelques obliques et de coulures rouges très lisibles. Points focaux : les traînées rouges sur fond jaune en haut à gauche et les épaisseurs orangées très texturées à dr

### Practical applications : accessibility and pedagogy

GPT-5 excels in two concrete use cases:
- **Alt text** : automatically generate accessibility descriptions for web images
- **Pedagogical analysis** : use multimodal vision to create educational content from images


### Reading the result — alt text generation

The previous cell calls `analyze_image_with_gpt5()` with a specialized accessibility prompt that imposes three constraints: (1) output ≤ 125 characters, (2) factual and objective tone, (3) no additional formatting. The output obtained is *Vue aérienne de la Gran Vía à Madrid au coucher du soleil, dôme du Metropolis, immeubles illuminés et circulation.* — **114 characters** (under the 125 threshold), validated by the `print` that follows.

**Why this length matters** : the WCAG 2.1 standard (criterion 1.1.1 *Non-text Content*) recommends short alt texts for informative images. Beyond 125 characters, screen readers truncate and the experience degrades. Below 50, the user loses context. The guided example in cell 19 below runs the three length levels to study this comfort zone — this is one of the uses where native multimodality replaces a separate pipeline (vision model → text model → formatting).

In [10]:
# Génération de descriptions d'accessibilité (Alt text)
if generate_alt_text and analysis_results:
    print("\n♿ GÉNÉRATION DE DESCRIPTIONS D'ACCESSIBILITÉ")
    print("=" * 55)
    
    # Prompt spécialisé pour l'accessibilité
    accessibility_prompt = f"""Génère une description d'accessibilité (alt text) pour cette image en {language}.
    
Critères :
- Maximum 125 caractères
- Description factuelle et objective
- Inclut les éléments essentiels pour la compréhension
- Évite les interprétations subjectives
- Adapté aux lecteurs d'écran

Format : Fournis UNIQUEMENT la description, sans formatage supplémentaire."""
    
    # Génération de l'alt text
    alt_result = analyze_image_with_gpt5(
        image_source=selected_example['url'],
        prompt=accessibility_prompt
    )
    
    if alt_result['success']:
        alt_text = alt_result['analysis'].strip()
        
        print(f"✅ Description d'accessibilité générée")
        print(f"📝 Alt text ({len(alt_text)} caractères) :")
        print(f'   "{alt_text}"')
        
        if len(alt_text) > 125:
            print(f"⚠️  Longueur dépassée ({len(alt_text)}/125 chars) - considérez une version plus courte")
        else:
            print(f"✅ Longueur optimale ({len(alt_text)}/125 chars)")
    else:
        print(f"❌ Erreur génération alt text: {alt_result['error']}")
else:
    print(f"\n⏭️  Génération alt text désactivée ou pas d'analyse disponible")


♿ GÉNÉRATION DE DESCRIPTIONS D'ACCESSIBILITÉ

🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://images.unsplash.com/photo-1539037116277-4db20889f2d4?w=800


✅ Description d'accessibilité générée
📝 Alt text (122 caractères) :
   "Vue aérienne de la Gran Vía à Madrid au coucher du soleil, avec le bâtiment Metrópolis à gauche et des voitures circulant."
✅ Longueur optimale (122/125 chars)


### Guided example: multi-level accessibility descriptions

*Contribution from a group of EPF students (TP2, PR #20551), integrated as a guided example.*

Web accessibility requires alternative texts (alt text) adapted to the context. The same image may require descriptions of different lengths depending on the use: quick navigation (≤ 125 characters), detailed description for screen readers (200-400 characters), or a pedagogical summary. This example generates the three levels with three distinct prompts — each one explicitly sets the target length and the intended audience — then measures in code the actual length of each text produced.

**Key points**:
- Each level has its **own prompt**: the maximum length and the audience are spelled out in the instruction
- The check relies on `len()` of the actual texts — compliance with the WCAG limit is **observed**, not assumed
- The short text targets ≤ 125 characters: beyond that, screen readers truncate (see the WCAG analysis above)

In [11]:
# Exemple guide : descriptions d'accessibilite multi-niveaux
# (contribution d'un groupe d'etudiants EPF, TP2, PR #20551)

# Etape 1 : Choix de l'image
mon_image = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSm6AMs5m9c7Nmp-IB1HUY4LeA4EjTF8sUTmW3JU-26cs1BF0cJuaFBV8c&s=10"  # image choisie par le groupe

# Etape 2 : Prompt pour alt text court (<= 125 caracteres)
prompt_court = "Fournissez une description factuelle et objective de l'image, en maximum 125 caractères, sans interprétation."

# Etape 3 : Prompt pour description detaillee (200-400 caracteres)
prompt_detaille = "Fournissez une description détaillée de l'image, en 200 à 400 caractères, pour les lecteurs d'écran. Incluez les éléments essentiels, les couleurs, la disposition et les actions visibles."
# Contraintes imposees : elements essentiels, couleurs, disposition, actions visibles

# Etape 4 : Prompt pour resume pedagogique niveau "secondary"
prompt_pedago = "Fournissez un résumé pédagogique de l'image, adapté à des élèves du secondaire. Utilisez un vocabulaire accessible et posez des questions de réflexion."

# Generation des trois niveaux
resultats = {}
for nom, prompt in [("court", prompt_court), ("detaille", prompt_detaille), ("pedago", prompt_pedago)]:
    r = analyze_image_with_gpt5(mon_image, prompt=prompt)
    if r['success']:
        resultats[nom] = r['analysis']
        print(f"\n--- {nom.upper()} ({len(r['analysis'])} chars) ---")
        print(r['analysis'][:300])

# Etape 5 : Verification des contraintes de longueur
print("\n Vérification des résultats")
print("Nombre de caractères dans le texte court:", len(resultats.get("court", "")))
print("Nombre de caractères dans la description détaillée:", len(resultats.get("detaille", "")))

print("\nRésultats disponibles pour les trois niveaux : court, détaillé et pédagogique:")
print("Texte court (<= 125 chars):", resultats.get("court", "Non généré"))
print("Description détaillée (200-400 chars):", resultats.get("detaille", "Non généré"))
print("Résumé pédagogique:", resultats.get("pedago", "Non généré"))



🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSm6AMs5m9c7Nmp-IB1HUY4LeA4EjTF8sUTmW3JU-26cs1B...



--- COURT (93 chars) ---
Personnage 3D en tenue sombre tenant un pistolet, gants noirs, cheveux bruns, regard de côté.

🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSm6AMs5m9c7Nmp-IB1HUY4LeA4EjTF8sUTmW3JU-26cs1B...



--- DETAILLE (294 chars) ---
Plan taille d’un personnage masculin en 3D, regard dur, cheveux bruns mi-longs. Il porte une tenue tactique bleu nuit et noire, gants et harnais. Main droite levée tenant un pistolet près du visage; un fusil est accroché en bandoulière. Éclairage latéral froid, fond industriel sombre en métal.

🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSm6AMs5m9c7Nmp-IB1HUY4LeA4EjTF8sUTmW3JU-26cs1B...



--- PEDAGO (1789 chars) ---
Résumé pédagogique
- L’image montre un personnage de fiction en 3D, habillé en tenue tactique sombre. Il tient un pistolet et adopte une pose sûre de lui, comme s’il était prêt pour une mission.
- La lumière met en valeur son visage et surtout son bras, ce qui renforce l’idée de force et de tension.

 Vérification des résultats
Nombre de caractères dans le texte court: 93
Nombre de caractères dans la description détaillée: 294

Résultats disponibles pour les trois niveaux : court, détaillé et pédagogique:
Texte court (<= 125 chars): Personnage 3D en tenue sombre tenant un pistolet, gants noirs, cheveux bruns, regard de côté.
Description détaillée (200-400 chars): Plan taille d’un personnage masculin en 3D, regard dur, cheveux bruns mi-longs. Il porte une tenue tactique bleu nuit et noire, gants et harnais. Main droite levée tenant un pistolet près du visage; un fusil est accroché en bandoulière. Éclairage latéral froid, fond industriel sombre en métal.
Résu

The demonstration analyses are complete. The interactive mode below lets you experiment with your own images and personalized analysis prompts.


### Reading the result — batch versus interactive mode

The previous cell tries `input()` to retrieve a URL or a local path. In Papermill / batch execution, standard input is not available: the `except EOFError` intercepts cleanly and switches to the message `⏭️  Mode interactif non disponible (exécution automatisée)`. This pattern is the repository's convention for notebooks that mix interactive and batch use: the cell remains functional in both modes without `raise NotImplementedError` (forbidden by C.1 — `pass`/`print` are the correct stubs).

The next cell moves on to the application use cases — this is the bridge between the technical demonstration and the business use. The 5 domains listed (History, Science, Geography, Art, Medicine) cover the disciplines where multimodal analysis brings the most versus a purely textual model.

In [12]:
# Mode interactif - Analyse d'image personnalisée
if notebook_mode == "interactive" and not skip_widgets:
    print("\n🎨 MODE INTERACTIF - ANALYSE PERSONNALISÉE")
    print("=" * 55)
    
    print("\n💡 Analysez votre propre image avec GPT-5:")
    print("Formats supportés: URL https:// ou chemin local")
    print("(Laissez vide pour passer à la suite)")
    
    try:
        user_image = input("\n🖼️  URL ou chemin de votre image: ").strip()

        if user_image:
            # Paramètres d'analyse personnalisés
            print("\n⚙️  Paramètres d'analyse (appuyez Entrée pour défaut):")
            custom_mode = input(f"📊 Mode [{analysis_mode}]: ").strip() or analysis_mode
            custom_prompt = input("📝 Prompt personnalisé (optionnel): ").strip()

            print(f"\n🔍 Analyse de votre image en cours...")

            # Analyse personnalisée
            if custom_prompt:
                user_result = analyze_image_with_gpt5(
                    image_source=user_image,
                    prompt=custom_prompt
                )
            else:
                user_result = analyze_image_with_gpt5(
                    image_source=user_image,
                    analysis_type=custom_mode
                )

            if user_result['success']:
                print(f"\n🎉 Analyse réussie!")
                print(f"⏱️  Temps: {user_result['metadata']['response_time']:.2f}s")
                print(f"\n📝 **Analyse GPT-5:**")
                print(user_result['analysis'])

                # Option de sauvegarde
                if export_analysis:
                    try:
                        save_choice = input("\n💾 Sauvegarder cette analyse ? (o/N): ").strip().lower()
                        if save_choice in ['o', 'oui', 'y', 'yes']:
                            output_dir = current_path / 'outputs' / 'gpt5_analysis'
                            output_dir.mkdir(parents=True, exist_ok=True)

                            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
                            analysis_file = output_dir / f"gpt5_analysis_{timestamp}.json"

                            with open(analysis_file, 'w', encoding='utf-8') as f:
                                json.dump(user_result, f, indent=2, ensure_ascii=False)

                            print(f"💾 Analyse sauvegardée: {analysis_file}")
                    except Exception as save_err:
                        if "StdinNotImplemented" in type(save_err).__name__:
                            print("\n⏭️  Sauvegarde automatique ignorée en mode batch")
                        else:
                            raise
            else:
                print(f"\n❌ Erreur: {user_result['error']}")
                print(f"🔍 Source: {user_result['image_source']}")
        else:
            print("\n⏭️  Mode interactif ignoré")

    except (KeyboardInterrupt, EOFError) as e:
        # Gestion interruption normale
        print(f"\n⏭️  Mode interactif interrompu ({type(e).__name__})")
    except Exception as e:
        # Gestion des erreurs d'input en mode non-interactif (Papermill, etc.)
        error_type = type(e).__name__
        if "StdinNotImplemented" in error_type or "input" in str(e).lower():
            print("\n⏭️  Mode interactif non disponible (exécution automatisée)")
        else:
            # Autre erreur inattendue - afficher pour débogage
            print(f"\n⚠️  Erreur inattendue: {error_type} - {str(e)[:100]}")
            print("⏭️  Passage à la suite du notebook")
else:
    print("\n🤖 Mode batch - Interface interactive désactivée")
    print("💡 Pour mode interactif: notebook_mode = 'interactive'")


🎨 MODE INTERACTIF - ANALYSE PERSONNALISÉE

💡 Analysez votre propre image avec GPT-5:
Formats supportés: URL https:// ou chemin local
(Laissez vide pour passer à la suite)

⏭️  Mode interactif non disponible (exécution automatisée)


With the interactive mode handled, the next cell presents the main application use cases of visual analysis by GPT-5 in a professional and pedagogical context.


### Reading the result — domain × prompt matrix

The previous cell catalogs five application domains (History, Science, Geography, Art and Culture, Medicine) with for each a predefined `prompt_template` that can be passed directly to `analyze_image_with_gpt5()`. Each template imposes a disciplinary role, an output format, and a tone suited to the domain — for example, the `Médecine` prompt specifies *Analyse cette image médicale pour des étudiants en médecine. Explique l'anatomie visible et les points d'intérêt clinique.*

The output contains no visible GPT-5 output — the cell only *announced* the use cases, without executing them yet. The guided example that follows runs one of them for geography, and the exercises at the end of the notebook ask to recombine these ingredients. This is an interesting pedagogical inversion: cell 24 shows the templates, the exercises ask to recombine them.

In [13]:
# Cas d'usage pédagogiques avec GPT-5
print("\n🎓 CAS D'USAGE PÉDAGOGIQUES GPT-5")
print("=" * 45)

# Exemples d'applications éducatives
educational_use_cases = {
    "Histoire": {
        "description": "Analyse de documents historiques, œuvres d'art, monuments",
        "exemple": "Analyser une fresque renaissance pour comprendre le contexte social",
        "prompt_template": "Analyse cette image historique pour des étudiants en histoire. Explique le contexte historique, les éléments symboliques, et l'importance culturelle."
    },
    
    "Sciences": {
        "description": "Analyse d'expériences, schémas scientifiques, phénomènes naturels",
        "exemple": "Expliquer un diagramme de cellule ou une réaction chimique",
        "prompt_template": "En tant qu'enseignant de sciences, explique cette image scientifique. Identifie les éléments techniques et leur fonctionnement."
    },
    
    "Géographie": {
        "description": "Étude de paysages, cartes, phénomènes géologiques",
        "exemple": "Analyser une photo satellite ou un paysage géographique",
        "prompt_template": "Analyse cette image géographique pour des étudiants. Explique les formations géologiques, le climat, et l'impact humain visible."
    },
    
    "Art et Culture": {
        "description": "Critique artistique, analyse stylistique, histoire de l'art",
        "exemple": "Comprendre les techniques d'un tableau impressionniste",
        "prompt_template": "Fais une analyse artistique de cette œuvre pour des étudiants en art. Inclus style, techniques, et signification culturelle."
    },
    
    "Médecine": {
        "description": "Analyse d'imagerie médicale, anatomie, cas cliniques",
        "exemple": "Expliquer une radiographie ou un schéma anatomique",
        "prompt_template": "Analyse cette image médicale pour des étudiants en médecine. Explique l'anatomie visible et les points d'intérêt clinique."
    }
}

# Affichage des cas d'usage
for domain, info in educational_use_cases.items():
    print(f"\n📚 **{domain}**")
    print(f"   📋 {info['description']}")
    print(f"   💡 Exemple: {info['exemple']}")
    print(f"   📝 Template de prompt disponible")

print(f"\n🎯 Avantages GPT-5 pour l'éducation:")
print(f"• **Multimodal** : Analyse texte + image simultanée")
print(f"• **Contextuel** : Comprend le niveau éducatif")
print(f"• **Adaptable** : Ajuste le vocabulaire selon l'audience")
print(f"• **Interactif** : Permet les questions de suivi")
print(f"• **Précis** : Analyse détaillée et factuelle")

print(f"\n📝 Exemple de workflow pédagogique:")
print(f"1. Sélection d'image pertinente au cours")
print(f"2. Analyse GPT-5 avec prompt éducatif")
print(f"3. Génération de questions pour les étudiants")
print(f"4. Discussion interactive basée sur l'analyse")
print(f"5. Évaluation de la compréhension")


🎓 CAS D'USAGE PÉDAGOGIQUES GPT-5

📚 **Histoire**
   📋 Analyse de documents historiques, œuvres d'art, monuments
   💡 Exemple: Analyser une fresque renaissance pour comprendre le contexte social
   📝 Template de prompt disponible

📚 **Sciences**
   📋 Analyse d'expériences, schémas scientifiques, phénomènes naturels
   💡 Exemple: Expliquer un diagramme de cellule ou une réaction chimique
   📝 Template de prompt disponible

📚 **Géographie**
   📋 Étude de paysages, cartes, phénomènes géologiques
   💡 Exemple: Analyser une photo satellite ou un paysage géographique
   📝 Template de prompt disponible

📚 **Art et Culture**
   📋 Critique artistique, analyse stylistique, histoire de l'art
   💡 Exemple: Comprendre les techniques d'un tableau impressionniste
   📝 Template de prompt disponible

📚 **Médecine**
   📋 Analyse d'imagerie médicale, anatomie, cas cliniques
   💡 Exemple: Expliquer une radiographie ou un schéma anatomique
   📝 Template de prompt disponible

🎯 Avantages GPT-5 pour l'éducat

### Guided example: three-mode personalized analysis for a domain

*Contribution from a group of EPF students (TP2, PR #20551), integrated as a guided example.*

Synthesis of the two previous examples: the `analyse_complete()` function chains three analyses of the same image — a short factual description, a detailed description for screen readers, a pedagogical summary contextualized by the domain — and returns everything in a dictionary displayed with `pprint`. The domain (`Géographie`), the level (`university`) and the image are set at the top of the cell.

**Key points**:
- A single function call produces the **three levels** of description — the result is a reusable dictionary, not three separate displays
- The educational prompt is **built from the domain**: changing `domain` at the top of the cell adapts the analysis to another discipline

In [14]:
# Exemple guide : analyse personnalisee a trois modes pour un domaine
# (contribution d'un groupe d'etudiants EPF, TP2, PR #20551)

# 1. Sélection du domaine et configuration
domain = "Géographie"
niveau = "university"  # Niveau éducatif cible

# 2. Image à analyser (URL publique)
image_source = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRPQ_gGhRNQdk5gxBqyA9diGOVps2ZliOpWX2ifjfPSPg&s=10"  # carte de France (choix du groupe)

# 3. Création du prompt personnalisé (inspiré des prompt_template de educational_use_cases)
custom_prompt = f"""Tu es un professeur expert en {domain}.
Analyse cette image pour des étudiants de niveau {niveau}.
Inclus :
1. Description accessible du sujet
2. Éléments techniques importants à observer
3. Contexte historique ou scientifique
4. 3 questions de réflexion pour les étudiants
"""

# 4. Analyse avec les 3 modes de description
def analyse_complete(image_url, prompt, domain):
    """
    Analyse une image avec les 3 modes et retourne les résultats.

    Args:
        image_url: URL ou chemin de l'image
        prompt: Prompt personnalisé pour le mode educational
        domain: Domaine d'étude pour le contexte

    Returns:
        Dict avec les 3 analyses (quick, detailed, educational)
    """



    quick = "Fournissez une description factuelle et objective de l'image, en maximum 125 caractères, sans interprétation."
    detailed = "Fournissez une description détaillée de l'image, en 200 à 400 caractères, pour les lecteurs d'écran. Incluez les éléments essentiels, les couleurs, la disposition et les actions visibles."
    educational = f"""Fournissez un résumé pédagogique de l'image, adapté à des élèves du {domain}. Utilisez un vocabulaire accessible et posez des questions de réflexion."""

    results = {}
    for nom, prompt in [("quick", quick), ("detailed", detailed), ("educational", educational)]:
        r = analyze_image_with_gpt5(image_source, prompt=prompt)
        if r['success']:
          results[nom] = r['analysis']
    return results

# 5. Exécuter l'analyse et afficher les résultats
import pprint
results = analyse_complete(image_source, custom_prompt, domain)
pprint.pprint(results)



🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRPQ_gGhRNQdk5gxBqyA9diGOVps2ZliOpWX2ifjfPSPg&s...



🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRPQ_gGhRNQdk5gxBqyA9diGOVps2ZliOpWX2ifjfPSPg&s...



🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRPQ_gGhRNQdk5gxBqyA9diGOVps2ZliOpWX2ifjfPSPg&s...


{'detailed': 'Carte politique de la France métropolitaine: territoire en rose '
             'saumon, mers en bleu. Départements et anciennes régions nommés; '
             'grandes villes signalées (Paris, Lyon, Marseille, Bordeaux, '
             'Lille, Toulouse, Nice, Nantes, Strasbourg). Frontières avec '
             'Belgique, Luxembourg, Allemagne, Suisse, Italie, Espagne. '
             'Atlantique à l’ouest, Manche au nord‑ouest, Méditerranée au '
             'sud‑est. Encart Paris et Corse; échelle en bas.',
 'educational': 'Résumé pédagogique de l’image\n'
                '\n'
                'Ce que montre la carte\n'
                '- C’est une carte de la France et de ses voisins européens. '
                'On y voit les régions historiques, les grandes villes, les '
                'mers et océans, les montagnes et les grands fleuves.\n'
                '- La France a trois façades maritimes: la Manche au '
                'nord-ouest, l’océan Atlantique à l’ouest e

## 🎯 Summary and Best Practices

### ✅ What you have learned

- [ ] **GPT-5 configuration** : OpenRouter API and optimal parameters
- [ ] **Multimodal analysis** : Combining text + image
- [ ] **Educational prompts** : Adapting to the teaching level
- [ ] **Pedagogical use cases** : Concrete applications by domain
- [ ] **Optimization** : Quality and performance parameters

### 🚀 Next steps

1. **Experiment** with your own educational images
2. **Test** different analysis levels depending on your audience
3. **Integrate** GPT-5 into your pedagogical workflows
4. **Combine** with DALL-E 3 for generation + analysis
5. **Explore** the advanced notebooks (Module 02)

### 💡 Usage tips

**✅ Best practices:**
- Use high-quality images
- Adapt the prompt to the educational level
- Combine visual and contextual analysis
- Generate pedagogical questions
- Save successful analyses

**❌ Avoid:**
- Images that are too small or of poor quality
- Vague or generic prompts
- Forgetting the educational context
- Ignoring the model's limitations
- Not checking factuality

### 🔗 Additional resources

- **OpenRouter documentation** : [openrouter.ai](https://openrouter.ai)
- **GPT-5 guide** : Multimodal capabilities
- **Educational templates** : `docs/genai-phase2-templates.md`
- **CoursIA standards** : `docs/genai-images-development-standards.md`

### 📖 Scholarly references

- **OpenAI** (2025). *GPT-5 System Card & Model Page*. — Multimodal model (text + vision), **400,000 token** context window, 128,000 maximum output tokens. Official sources: openai.com/gpt-5/ and developers.openai.com/api/docs/models/gpt-5.
- **OpenAI** (2024). *GPT-4o System Card*. — Multimodal precursor of GPT-5 (vision + text unified), basis of the multimodal API used via OpenRouter in this notebook.


### Reading the result — what the next cell really presents

The `print('Cas d'usage pédagogiques')` above is a **static catalog**, not an execution. The five domains listed are there to give the student a structured starting point — each `prompt_template` is immediately reusable via `analyze_image_with_gpt5(image, prompt=...)`, without having to rewrite it. The `print('Template de prompt disponible')` is the clue that the cell has already played its pedagogical role by the time the student reads it, even if nothing was executed.

**Three things to remember for the summary cell** :
1. The OpenRouter API exposes 26 distinct GPT-5 models — the nomenclature is wider than the official OpenAI nomenclature, and the selection is made on the `contexte × coût × capabilities` trade-off (vision, tools, etc.).
2. The three analysis modes (quick/detailed/educational) are not interchangeable: quick is optimized for WCAG alt-text, educational produces student-guiding questions — it is the only mode that provides a turnkey pedagogical deliverable.
3. The multimodal pipeline (encode → POST → parse) is uniform whatever the image — the complexity lies in the prompt, not in the transport.

***

## Exercises to complete

The following three exercises extend the guided examples of this notebook: sensitivity of the analysis to the wording of the prompt, adapting an alt text to its publication context, and comparing two images from the same domain. Each skeleton runs without error as is — it is up to you to fill in the `TODO etudiant`.

### Exercise 1 : Neutral prompt vs oriented prompt — measuring the sensitivity to wording

The same image analyzed by GPT-5 can produce very different answers depending on how the prompt is worded. The goal is to **measure** this sensitivity instead of undergoing it: two prompts that differ only in their angle (neutral vs oriented toward a reading frame), same image, same output format.

**Steps**:
1. Choose an image from `example_images`
2. Write a `prompt_neutre`: objective description, without theoretical orientation
3. Write a `prompt_oriente`: same output format, but it orients the analysis toward a frame (historical, aesthetic, technical...)
4. Run both analyses and compare: elements mentioned, vocabulary, structure
5. Conclude in one sentence: which wording serves which pedagogical use?

**Hint**: change ONLY the orientation instruction between the two prompts — if the output format also differs, the comparison no longer measures the wording.

In [15]:
# Exercice 1 : Prompt neutre vs prompt oriente -- mesurer la sensibilite de la formulation
# TODO etudiant : comparer deux analyses de la MEME image sous deux formulations

# Etape 1 : Choix de l'image
image_exo1 = example_images[1]['url']  # Laboratoire (Science et Technologie)

# Etape 2 : Redaction des deux prompts
prompt_neutre = "..."   # TODO etudiant : description objective, format de sortie en sections
prompt_oriente = "..."  # TODO etudiant : meme format, mais oriente vers UN cadre de lecture
# Indice : seul l'angle change -- le format de sortie reste identique

# Etape 3 : Lancement des deux analyses
# TODO etudiant : resultat_neutre = analyze_image_with_gpt5(image_exo1, prompt=prompt_neutre)
# TODO etudiant : resultat_oriente = analyze_image_with_gpt5(image_exo1, prompt=prompt_oriente)

# Etape 4 : Comparaison mesuree
# TODO etudiant : les deux analyses citent-elles les memes elements visuels ?
# TODO etudiant : quels termes apparaissent chez l'une et pas chez l'autre ?

resultat_neutre = None   # TODO etudiant : remplacer par l'appel a analyze_image_with_gpt5
resultat_oriente = None  # TODO etudiant

print("Exercice a completer")


Exercice a completer


### Exercise 2 : Contextual alt text — one image, three contexts

The accessibility guided example generated three description **lengths** for the same image. The next level: the same image published in three different **contexts** calls for three different alt texts, even if the 125-character limit stays the same. A relevant alt text depends on what the reader came looking for.

**Steps**:
1. Choose an image from `example_images` (or your own URL)
2. Define three publication contexts: popular-science article, social network, school textbook
3. Write one prompt per context: same ≤ 125-character limit, but the information angle adapts to the context
4. Generate the three texts and check: each one is ≤ 125 characters, and the three texts are indeed **distinct**

**Hint**: ask yourself "what did the reader come to do on this page?" — that question determines what to keep in the 125 characters.

In [16]:
# Exercice 2 : Alt text contextuel -- une image, trois contextes
# TODO etudiant : produire trois alt texts DIFFERENTS pour la meme image

# Etape 1 : Choix de l'image
image_exo2 = example_images[3]['url']  # Paysage naturel

# Etape 2 : Definition des trois contextes de publication
contexte_article = "article de vulgarisation scientifique"  # TODO etudiant : affiner le lectorat vise
contexte_social = "publication sur un reseau social"        # TODO etudiant
contexte_manuel = "manuel scolaire de geographie"           # TODO etudiant

# Etape 3 : Redaction d'un prompt par contexte
# TODO etudiant : chaque prompt exige une description <= 125 caracteres ADAPTEE au contexte
# Indice : meme limite partout -- seul l'angle d'information change

# Etape 4 : Generation et verification
# TODO etudiant : alt_texts = {contexte: texte} rempli par les appels a analyze_image_with_gpt5
# TODO etudiant : verifier len(texte) <= 125 pour chacun, et que les trois textes different

alt_texts = {}  # TODO etudiant

print("Exercice a completer")


Exercice a completer


### Exercise 3 : Image comparator — two images from the same domain side by side

**Estimated duration :** 15-20 minutes

**Objective**: compare two images from the same category in a single workflow — individual analyses with an identical format, then cross-comparison of the results.

**Instructions**:
1. **Choose a domain** among the pedagogical use cases presented above
2. **Select two images** relevant to this domain (public URLs or local images)
3. **Write an analysis prompt** that imposes an identical structure for both images
4. **Implement `compare_images()`**: analysis of each image, then cross-comparison — common points, differences, verdict
5. **Display the result** in a readable way (dictionary + `pprint`)

> **Hints:**
- Hint 1 : `analyze_image_with_gpt5()` accepts any URL — call it once per image
- Hint 2 : to cross-compare, inject the text of the first analysis into the prompt of the second — GPT-5 will compare the visible image to the provided text
- Hint 3 : the same output format for both analyses makes the comparison mechanical

In [17]:
# Exercice 3 : Comparateur d'images -- deux images du meme domaine cote a cote
# TODO etudiant : implementer compare_images() pour deux images d'un meme domaine

# 1. Selection des deux images
image_a = example_images[0]['url']  # TODO etudiant : premiere image du domaine choisi
image_b = example_images[3]['url']  # TODO etudiant : remplacer par une image du MEME domaine

# 2. Prompt d'analyse pour chaque image
# TODO etudiant : redigez un prompt qui impose une structure comparable entre les deux analyses
# Indice : meme format de sortie pour les deux images -- sinon la comparaison est arbitraire

# 3. Fonction de comparaison
def compare_images(url_a, url_b, prompt):
    """
    Analyse deux images du meme domaine et produit une comparaison structuree.

    Args:
        url_a: URL de la premiere image
        url_b: URL de la seconde image
        prompt: prompt d'analyse (format identique pour les deux images)

    Returns:
        Dict avec les deux analyses et leur comparaison
    """
    comparaison = {}

    # TODO etudiant : analyser chaque image individuellement avec analyze_image_with_gpt5
    # TODO etudiant : croiser les resultats -- points communs, differences, verdict
    # Indice : le texte de la premiere analyse peut etre injecte dans le prompt de la seconde

    pass  # Remplacer par votre implementation

    return comparaison

# 4. Executer et afficher
# Indice : pprint.pprint pour un affichage lisible du dict
resultat_comparaison = None  # TODO etudiant : compare_images(image_a, image_b, prompt)

print("Exercice a completer")


Exercice a completer


### Success criteria

- [ ] Both images belong to the same domain and the analyses use the same format
- [ ] `compare_images()` returns a dictionary with both analyses and the comparison
- [ ] The comparison separates common points and differences in a structured way
- [ ] The results are clearly presented (`pprint` or formatted display)
- [ ] The total response time stays reasonable (2 analyses + cross-comparison)

### Extension (optional)

For advanced students:
- Implement an **evaluation** of student answers with GPT-5
- Create an **interactive quiz** based on the generated questions
- Extend the comparator to **three images** or more (comparison matrix)


### Reading the result — exercise 3 (synthesis)

The previous cell poses a structuring synthesis exercise: choose a domain among the five cataloged in the *Pedagogical use cases* cell, select two images from this domain, impose an identical analysis format, and implement `compare_images()`, which cross-compares the results. The provided skeleton only contains `pass` and returns an empty dictionary: the notebook runs end to end as long as the exercise is not completed. Exercises 1 and 2, which are shorter, build on the guided examples in the body of the notebook: the first measures sensitivity to wording, the second adapts an alt text to its publication context.

## Conclusion

GPT-5 image generation (via OpenRouter) embodies the **native multimodal** approach: the model does not merely generate images from a text prompt, it analyzes and produces in a single flow, which opens three distinct pedagogical usage modes — quick (206 chars), detailed (2606 chars), educational (2908 chars). The inference cost / content richness ratio is directly exposed to the user by the choice of mode.

**Specific contribution of native multimodality**: the automatic generation of alt text (114 chars optimal, under the 125 chars threshold of the WCAG accessibility referentials) integrates image description as a **first-class deliverable**, not as an optional post-processing. For the 5 educational use cases covered (History, Science, Geography, Art and Culture, Medicine), this is a substantial accessibility gain over purely text-to-image models.

**Positioning in the ecosystem**: GPT-5 completes the course's palette by covering the *comprehension + generation* axis, where DALL-E 3 focuses on generation alone and SD XL Turbo exposes the internal workings. The 26 GPT-5 models detected via the OpenRouter API open a wide comparative experimentation ground but introduce a strong dependency on the routing provider.

**To go further**: explore the `04-1-Educational-Content-Generation` notebook which orchestrates several models to produce a complete pedagogical deliverable (image + alt text + structured metadata).